# Reranking

## Load Secrets

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## Hybrid Search Utility

In [7]:
import os
import cohere
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import FusionQuery, Prefetch, SparseVector

co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
qdrant = QdrantClient(
    api_key=os.environ["QDRANT_API_KEY"],
    url=os.environ["QDRANT_URL"],
)
bm25 = SparseTextEmbedding("Qdrant/bm25")
COLLECTION = "week2_article"



def search_hybrid(query: str, k: int=5):
    q_dense = co.embed(
        model="embed-v4.0",
        input_type="search_query",
        embedding_types=["float"],
        texts=[query],
    ).embeddings.float[0]

    sv = next(iter(bm25.embed(query)))
    q_sparse = SparseVector(indices=sv.indices.tolist(), values=sv.values.tolist())

    return qdrant.query_points(
        collection_name=COLLECTION,
        prefetch=[
            Prefetch(
                query=q_dense,
                using="dense",
                limit=k,
            ),
            Prefetch(
                query=q_sparse,
                using="sparse",
                limit=k,
            ),
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    ).points


## Rerank After Search Utility

In [12]:
def search_with_rerank(query: str, k: int = 5, fetch: int = 20):
    candidates = search_hybrid(query, k=fetch)
    if not candidates:
        return []
    docs = [c.payload["text"] for c in candidates]
    rr = co.rerank(
        model="rerank-v3.5", query=query, documents=docs, top_n=min(len(candidates), k)
    )
    return [
        {"text": candidates[r.index].payload["text"], "score": r.relevance_score}
        for r in rr.results
    ]

## Evaluation

In [13]:
query = "what does the article say about evaluation and measurement?"

print("--- hybrid only (top 3 of 20) ---")
for h in search_hybrid(query, k=20)[:3]:
    snippet = h.payload["text"][:100].replace("\n", " ")
    print(f"  {snippet}…")

print("\n--- hybrid + rerank (top 3 of 5) ---")
for r in search_with_rerank(query, k=3):
    snippet = r["text"][:100].replace("\n", " ")
    print(f"  {r['score']:.3f}  {snippet}…")

--- hybrid only (top 3 of 20) ---
  ## **§8 - The Evaluation Harness**   **==> picture [414 x 170] intentionally omitted <==**  I'll kee…
  Vibes don't compose. Vibes don't survive a model update. Vibes can't tell you whether your last chan…
  There are roughly four flavors of eval, and you’ll need all of them eventually.   **Reference-based …

--- hybrid + rerank (top 3 of 5) ---
  0.060  ## **§8 - The Evaluation Harness**   **==> picture [414 x 170] intentionally omitted <==**  I'll kee…
  0.058  - **Trajectory evals.** Score the _path_ , not just the final answer. Did the agent take 3 steps or …
  0.052  There are roughly four flavors of eval, and you’ll need all of them eventually.   **Reference-based …


## Recap

- Rerank = cross-encoder that reads query+document together. Use it on a small candidate set after retrieval. Cohere rerank-v3.5 is what we ship.